# XAUUSD XGBoost Training

**What this does:**
1. Clones the repo
2. Installs dependencies
3. Gets the training data (export from API OR upload your zip)
4. Validates + sanitizes
5. Trains a 42-feature XGBoost model
6. Downloads the artifact

---
**Step 1: Run ▶ the cells below in order**

In [ ]:
# 1. Clone the repo
!git clone https://github.com/webkaave/tradebot.git
%cd tradebot

In [ ]:
# 2. Install everything (includes XGBoost, Optuna, sklearn)
!pip install -e ".[train]" -q
print("Done")

---
**Step 2: Get the training data**

Choose ONE of the two cells below (A or B), then run it.

### Option A: Auto-export (need a Twelve Data API key)
[Get a free key here](https://twelvedata.com/apikey) — takes 30 seconds.

Run the cell below, paste your key when prompted, and it downloads everything.

In [ ]:
# A — Export from API
API_KEY = input("Paste your Twelve Data API key: ")
import os
os.environ["TWELVE_DATA_API_KEY"] = API_KEY

print("\nExporting XAUUSD M15/H1/H4 + DXY + EUR/USD...")
!python scripts/export_training_bundle.py --years 5 --output-dir data/training

print("\nExporting US10Y yields...")
!python scripts/export_fred_series.py --output data/training/us10y_daily.csv

!ls -lh data/training/

### Option B: Upload your own zip

If you already have the CSVs on your computer, zip them and upload here.

**On your computer:**
```bash
cd data && zip -r training_bundle.zip training/ && cd ..
```

Then run the cell below and select the zip file.

In [ ]:
# B — Upload zip
from google.colab import files
print("Select your training_bundle.zip file:")
uploaded = files.upload()

!mkdir -p data/training
!unzip -o training_bundle.zip -d data/training/
!ls -lh data/training/

---
**Step 3: Validate, sanitize, and train**

In [ ]:
# 3. Validate every CSV
!python scripts/validate_training_data.py --csv data/training/xauusd_m15.csv
!python scripts/validate_training_data.py --csv data/training/xauusd_h1.csv
!python scripts/validate_training_data.py --csv data/training/xauusd_h4.csv
!python scripts/validate_training_data.py --csv data/training/eurusd_m15.csv
!python scripts/validate_training_data.py --csv data/training/dxy_m15.csv
!python scripts/validate_training_data.py --csv data/training/us10y_daily.csv
print("\nAll valid")

In [ ]:
# 4. Sanitize (fix any bad high/low values)
import glob
for f in sorted(glob.glob('data/training/*.csv')):
    !python scripts/sanitize_training_data.py "$f"
print("Done")

In [ ]:
# 5. Train the model (this takes ~30-60 min)
!python scripts/train_xgboost.py \
  --m15 data/training/xauusd_m15.csv \
  --h1 data/training/xauusd_h1.csv \
  --h4 data/training/xauusd_h4.csv \
  --dxy data/training/eurusd_m15.csv \
  --us10y data/training/us10y_daily.csv \
  --real-dxy data/training/dxy_m15.csv \
  --output models/xgb_xauusd_v1.pkl \
  --trials 50 \
  --binary

In [ ]:
# 6. Check the model
import joblib
a = joblib.load('models/xgb_xauusd_v1.pkl')
print(f"Type:        {a.get('artifact_type')}")
print(f"Features:    {len(a.get('feature_columns', []))} columns")
print(f"Threshold:   {a.get('threshold')}")
print(f"Model:       {type(a.get('model')).__name__}")

In [ ]:
# 7. Download the model to your computer
from google.colab import files
files.download('models/xgb_xauusd_v1.pkl')

---
## After download — on your local machine

```bash
# Move the new model into place
mv ~/Downloads/xgb_xauusd_v1.pkl models/overlap_macro_trend_xgb.pkl

# Quick test
python scripts/dry_model_signal.py --ignore-calendar
```

That's it. No config changes needed.